# AXIS-Y 비건 콜라겐 아이 세럼 아마존 리뷰 시계열 데이터 정제 및 인사이트 분석

- **분석 대상:** `AXIS-Y Vegan Collagen Eye Serum (ASIN: B0CWPCX1D2)` 아마존 글로벌 고객 리뷰
- **분석 기간:** `2024-06-01 ~ 2026-10-21` (총 478건, 활성 리뷰 발생일 324일, 126주, 26개월)
- **핵심 목표:**
  1. 비정형/반정형 아마존 리뷰 데이터를 일별·주별·월별 시계열 데이터로 정제
  2. 결측치(`title` 125건, `body` 2건) 및 이상치(`reviewer_name` 오입력 4건), 층화 표집 편향(Sampling Bias) 보정
  3. 이동평균(MA), 전월 대비 변화율(MoM), 구간별 변동성($\sigma$), 시계열 분해(Decomposition) 및 베이스라인 예측 수행

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_clean = pd.read_csv('data/reviews_cleaned.csv')
daily_df = pd.read_csv('data/timeseries_daily.csv')
weekly_df = pd.read_csv('data/timeseries_weekly.csv')
monthly_df = pd.read_csv('data/timeseries_monthly.csv')

print('정제 완료 리뷰 데이터:', df_clean.shape)
print('일별 시계열 데이터 (활성일):', daily_df.shape)
print('주별 시계열 데이터 (연속 주):', weekly_df.shape)
print('월별 시계열 데이터 (활성 월):', monthly_df.shape)

정제 완료 리뷰 데이터: (478, 33)
일별 시계열 데이터 (활성일): (324, 18)
주별 시계열 데이터 (연속 주): (126, 13)
월별 시계열 데이터 (활성 월): (26, 30)


## 1. 데이터 정제 및 결측치·이상치·표집 편향 검증
- `title` 결측(125건): 비영어권 마켓플레이스(84건) 및 미국 모바일 간이 리뷰(41건) 특성 확인 후 빈 문자열 표준화
- `body` 결측(2건): 제목만 존재하는 리뷰(`REV_0288`) 및 별점만 존재하는 리뷰(`REV_0367`) 플래그 처리
- `reviewer_name` 이상치(4건): 35자 초과 리뷰 문장이 닉네임에 삽입된 케이스 익명 표준화
- **층화 표집 가중치(`pop_weight`) 산출**: 모집단(4,325건, 평균 4.44★) 대비 수집 표본(478건, 단순평균 3.19★)의 별점별 표집 비율 역수를 가중치로 부여

In [2]:
# 별점별 모집단 보정 가중치 및 핵심 토픽/불만 언급률 확인
summary_by_star = df_clean.groupby('rating_num')[[
    'pop_weight', 'topic_cooling_roller', 'topic_hydration',
    'issue_skin_irritation', 'issue_packaging_pump', 'issue_no_effect'
]].mean().round(3)
print(summary_by_star)

            pop_weight  topic_cooling_roller  topic_hydration  issue_skin_irritation  issue_packaging_pump  issue_no_effect
rating_num                                                                                                                 
1.0              0.191                 0.130            0.110                  0.310                 0.180            0.230
2.0              0.319                 0.283            0.150                  0.250                 0.133            0.167
3.0              0.407                 0.319            0.128                  0.128                 0.074            0.106
4.0              0.579                 0.354            0.273                  0.040                 0.030            0.051
5.0              2.753                 0.352            0.272                  0.000                 0.032            0.000


## 2. 월별 시계열 추이 확인 (2025-01 ~ 2026-10 연속 구간)
- 월별 리뷰 유입량(`review_count`), 3개월 이동평균(`ma3_count`), 전월 대비 증감률(`mom_growth_pct`)
- 표본 단순평균 평점(`mean_rating`) vs 모집단 가중 보정 평점(`weighted_rating`)

In [3]:
cols_view = ['year_month', 'review_count', 'ma3_count', 'mom_growth_pct', 'mean_rating', 'weighted_rating', 'global_share', 'irritation_cnt', 'packaging_cnt']
print(monthly_df[monthly_df['year_month'] >= '2025-01'][cols_view].to_string(index=False))

year_month  review_count  ma3_count  mom_growth_pct  mean_rating  weighted_rating  global_share  irritation_cnt  packaging_cnt
   2025-01             3       2.33             0.0        4.667            4.905           0.0               0              1
   2025-02             7       4.33           133.3        2.714            3.230          14.3               1              0
   2025-03             9       6.33            28.6        2.667            4.009           0.0               3              1
   2025-04             7       7.67           -22.2        2.429            3.005          14.3               1              2
   2025-05            12       9.33            71.4        2.917            3.336           0.0               1              1
   2025-06             7       8.67           -41.7        1.714            2.199          28.6               3              1
   2025-07            15      11.33           114.3        2.533            2.829          20.0               3

## 3. 전체 파이프라인 재현 실행
- 아래 셀을 실행하면 `analysis.py`의 정제·집계·시각화(차트 6종 생성) 파이프라인 전체가 동일하게 재현됩니다.

In [4]:
import analysis
analysis.main()

data/reviews_cleaned.csv, timeseries_daily.csv, timeseries_weekly.csv, timeseries_monthly.csv 및 images/01~06.png 생성 완료
